# Kaggle Smoke Test — Gemma4-E4B QLoRA (T4 x2, 10 samples)

**Purpose:** Test `lightning_gemma_e4b_qlora_80gb.ipynb` logic on **Kaggle `2×T4 16GB`** `D:\wasp\datascience\lightning_gemma_e4b_qlora_80gb.ipynb:26` before moving to `A100-80GB`/`B200`. Uses **same** `google/gemma-4-E4B-it` `D:\wasp\datascience\lightning_gemma_e4b_qlora_80gb.ipynb:32` `QLoRA 4-bit nf4 r64` but **only 10 samples** `train 10 → val 10` to verify `processor` `collate_fn` `forward` `generate` without burning credits.

**HF:** `ShivRamSaud/gemma4-e4b-qlora-astroclimb` — same `HF_REPO_ID` as Lightning. **Token:** Add `HF_TOKEN` in `Kaggle → Secrets → Add Secret → Key: HF_TOKEN → Value: hf_...` `D:\wasp\datascience\lightning_gemma_e4b_qlora_80gb.ipynb:146`.

**Run time:** `~3-5 min` on `T4 x2`. If `Forward OK loss` and `Generate OK` print, move to Lightning `A100`.


In [1]:
# --- 0. Config (Smoke: 2 samples, batch 1 — keep everything 1) ---
HF_REPO_ID = "ShivRamSaud/gemma4-e4b-qlora-astroclimb"
MODEL_ID = "google/gemma-4-E4B-it"
N_SMOKE = 2  # keep everything 1: just 2 rows
BATCH_SIZE = 1  # 1 as you asked
MAX_SEQ_LEN = 512  # smaller for T4 smoke
print(f"Smoke test: {MODEL_ID} on {N_SMOKE} samples, batch {BATCH_SIZE}, HF_REPO {HF_REPO_ID}")


Smoke test: google/gemma-4-E4B-it on 2 samples, batch 1, HF_REPO ShivRamSaud/gemma4-e4b-qlora-astroclimb


In [2]:
# --- 1. Setup (Kaggle T4 x2) — Gemma4 needs transformers>=5.15.1 ---
import os, json, time, re, gc, base64, sys, subprocess
from pathlib import Path
from io import BytesIO
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
print(f"torch {torch.__version__} cuda {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise SystemExit("Enable GPU T4 x2: Settings → Accelerator → GPU T4 x2")
assert torch.cuda.device_count() >= 2, "Need 2xT4 — Settings → GPU T4 x2"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_ALLOC_CONF set")
# Gemma4 (model_type gemma4) needs transformers>=5.15.1, Kaggle default is 5.0.0 — upgrade
import importlib.metadata as _im
try:
    _ver = _im.version("transformers")
    from packaging import version as _pv
    _need = _pv.parse(_ver) < _pv.parse("5.15.1")
    print(f"transformers {_ver} need upgrade (>=5.15.1 for gemma4): {_need}")
except: _need=True; _ver="0.0.0"
if _need:
    print("Upgrading transformers for Gemma4...")
    for _attempt in range(3):
        try:
            if _attempt==0:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/transformers.git", "accelerate"])
            elif _attempt==1:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=5.15.1", "accelerate"])
            else:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/huggingface/transformers.git", "accelerate"])
            print(f"install attempt {_attempt+1} OK")
            break
        except Exception as _e:
            print(f"install attempt {_attempt+1} failed {_e}")
            import time as _t; _t.sleep(5)
    for m in list(sys.modules.keys()):
        if m.startswith("transformers"): del sys.modules[m]
    import importlib as _il; _il.invalidate_caches()
import transformers; print(f"transformers {transformers.__version__} ready for gemma4")
try:
    import bitsandbytes; print(f"bitsandbytes {bitsandbytes.__version__}")
    import peft; print(f"peft {peft.__version__}")
except:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.46.1", "peft>=0.15.0"])
    import bitsandbytes, peft
    print("installed bitsandbytes/peft")
try:
    import seaborn as sns
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seaborn"])
    import seaborn as sns
    print("seaborn installed")


torch 2.10.0+cu128 cuda True
['Tesla T4', 'Tesla T4']
VRAM: 15.6 GB
PYTORCH_ALLOC_CONF set
transformers 5.0.0 need upgrade (>=5.15.1 for gemma4): True
Upgrading transformers for Gemma4...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.1 MB/s eta 0:00:00
install attempt 1 OK
transformers 5.17.0.dev0 ready for gemma4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.3 MB/s eta 0:00:00
installed bitsandbytes/peft


In [3]:
# --- 1b. HF Login (ShivRamSaud) ---
# Add HF_TOKEN in Kaggle: Secrets (right panel) → Add Secret → Key: HF_TOKEN → Value: hf_... (from huggingface.co/settings/tokens → Create Write token)
from huggingface_hub import login
import os
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print(f"Loaded HF_TOKEN from Kaggle Secrets ...{HF_TOKEN[-4:]}")
    except Exception as e:
        print(f"No HF_TOKEN in Secrets: {e}")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print(f"HF login OK for ShivRamSaud ...{HF_TOKEN[-4:]}")
else:
    print("No HF_TOKEN — will run smoke without push (add HF_TOKEN to Secrets to test push)")
print(f"HF_REPO_ID: {HF_REPO_ID}")


Loaded HF_TOKEN from Kaggle Secrets ...aUgL
HF login OK for ShivRamSaud ...aUgL
HF_REPO_ID: ShivRamSaud/gemma4-e4b-qlora-astroclimb


In [4]:
# --- 2. Data loading — HF dataset only (ShivRamSaud/astroclimb_train 10.1GB) ---
from datasets import load_dataset
from huggingface_hub import hf_hub_download
HF_TRAIN_DATASET = "ShivRamSaud/astroclimb_train"
TRAIN_CSV = f"hf://{HF_TRAIN_DATASET}/train.csv"
print(f"Loading HF dataset: {HF_TRAIN_DATASET} (no Kaggle fallback)")
try:
    ds = load_dataset(HF_TRAIN_DATASET, split="train")
    train = ds.to_pandas()
    print(f"Loaded from HF via load_dataset: {train.shape}")
except Exception as _e:
    print(f"load_dataset failed {_e}, trying hf_hub_download")
    csv_path = hf_hub_download(repo_id=HF_TRAIN_DATASET, filename="train.csv", repo_type="dataset")
    train = pd.read_csv(csv_path)
    print(f"Loaded from HF via hf_hub_download: {train.shape} {csv_path}")
    TRAIN_CSV = csv_path
label_cols = ["same_figure","same_paper","related_papers","unrelated_papers"]
train["label"] = train[label_cols].idxmax(axis=1)
train["label_id"] = train["label"].map({c:i for i,c in enumerate(label_cols)})
print(train["label"].value_counts().head())
def is_image_str(s):
    return isinstance(s, str) and len(s) > 200 and s.strip().startswith("iVBORw0KGgo")
def convert_str_to_PIL(s):
    return Image.open(BytesIO(base64.b64decode(s))).convert("RGB")
train["obj_1_is_img"] = train["obj_1"].apply(is_image_str)
train["obj_2_is_img"] = train["obj_2"].apply(is_image_str)
train["pair_type"] = train.apply(lambda r: ("IMG" if r["obj_1_is_img"] else "TXT") + "-" + ("IMG" if r["obj_2_is_img"] else "TXT"), axis=1)
print(train["pair_type"].value_counts())
# Smoke: take 10 diverse samples (not balanced, just quick)
smoke_df = train.sample(n=N_SMOKE, random_state=42).reset_index(drop=True)
print(f"smoke_df {smoke_df.shape}")
display(smoke_df[["label","pair_type"]].head())


Loading HF dataset: ShivRamSaud/astroclimb_train (no Kaggle fallback)


README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/10.1G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded from HF via load_dataset: (10000, 7)
label
same_paper          3000
related_papers      3000
unrelated_papers    3000
same_figure         1000
Name: count, dtype: int64
pair_type
TXT-IMG    4000
IMG-IMG    3000
TXT-TXT    3000
Name: count, dtype: int64
smoke_df (2, 12)


,label,pair_type
0,related_papers,TXT-TXT
1,related_papers,IMG-IMG


In [5]:
# --- 3. Prompt (same as Lightning D:\wasp\datascience\lightning_gemma_e4b_qlora_80gb.ipynb:221) ---
SYSTEM_PROMPT = """You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.

Classes:
- same_figure: The caption directly describes the figure in front of you. Visual elements (axes, labels, numbers, morphology) are mentioned verbatim in the text, or the text reads like \"Figure X shows...\" matching the image.
- same_paper: Same study, different figures. Similar writing style, same instruments/datasets/authors hinted in text, or visual style (fonts, colors, layout) is consistent, but NOT a direct caption-figure match.
- related_papers: Different papers where one builds on the other. Overlapping methods, shared datasets, or a figure/caption that looks like a cited prior result, but style/authors differ.
- unrelated_papers: No clear link. Different topics, instruments, scales, or writing/visual style with no overlap.

Base your decision only on visual and textual content. Do not assume same_figure is impossible for any pair type - judge from alignment.
Output ONLY the lowercase label (e.g., related_papers), no explanation, no punctuation."""
def build_user_content(row):
    parts=[]
    for col, name in [("obj_1","Object A"), ("obj_2","Object B")]:
        s=row[col]
        if row[f"{col}_is_img"]: parts.append({"name": name, "is_img": True, "pil": convert_str_to_PIL(s)})
        else: parts.append({"name": name, "is_img": False, "text": str(s)[:2000]})
    return parts
def build_messages(row):
    parts=build_user_content(row)
    content=[{"type":"text","text": SYSTEM_PROMPT+"\n"}]
    for p in parts:
        if p["is_img"]:
            im=p["pil"].copy(); im.thumbnail((448,448))
            content.append({"type":"image","image": im})
    for p in parts:
        if not p["is_img"]: content.append({"type":"text","text": "\n"+p["name"]+" (caption): "+p["text"]})
        else: content.append({"type":"text","text": "\n"+p["name"]+": [Figure image]"})
    content.append({"type":"text","text":"\nAnswer with one label:"})
    return [{"role":"user","content": content}]
print(SYSTEM_PROMPT[:300])
print(build_messages(smoke_df.iloc[0])[0]["content"][0]["text"][:200])


You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.

Classes:
- same_figure: The caption directly descri
You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what


In [6]:
# --- 4. Model (4-bit nf4, no LoRA for smoke — just test base forward) ---
from transformers import AutoProcessor, BitsAndBytesConfig
import torch
print(f"Loading {MODEL_ID} 4-bit for smoke test")
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left", trust_remote_code=True)
except TypeError:
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Processor: {type(processor).__name__}")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
try:
    from transformers import AutoModelForMultimodalLM as ModelClass
    print("Using AutoModelForMultimodalLM")
except:
    from transformers import AutoModelForImageTextToText as ModelClass
    print("Using AutoModelForImageTextToText")
model = ModelClass.from_pretrained(MODEL_ID, device_map="auto", quantization_config=bnb_config, max_memory={0:"14GB",1:"14GB"}, trust_remote_code=True)
print(f"Model loaded 4-bit: {type(model).__name__} on {model.device if hasattr(model, 'device') else 'auto'}")
print(f"Has generate: {hasattr(model, 'generate')}")
print("Smoke: no LoRA — just base 4-bit forward test (Lightning will use r64 LoRA)")


Loading google/gemma-4-E4B-it 4-bit for smoke test


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Processor: Gemma4Processor
Using AutoModelForMultimodalLM


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded 4-bit: Gemma4ForConditionalGeneration on cuda:1
Has generate: True
Smoke: no LoRA — just base 4-bit forward test (Lightning will use r64 LoRA)


In [7]:
# --- 5. Collator + Forward Test (2 samples, batch 1) ---
def row_to_messages_and_label(row):
    messages = build_messages(row)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    images=[]
    for turn in messages:
        for part in turn["content"]:
            if part.get("type")=="image": images.append(part["image"])
    return {"messages": messages, "text": text, "images": images, "label": row["label"]}
smoke_list = [row_to_messages_and_label(smoke_df.iloc[i]) for i in range(len(smoke_df))]
print(f"smoke_list {len(smoke_list)} sample label={smoke_list[0]['label']} images={len(smoke_list[0]['images'])}")
def collate_fn(features):
    batch_texts = [processor.apply_chat_template(f["messages"], tokenize=False, add_generation_prompt=False) + "\n" + f["label"] for f in features]
    flat_images = []
    for f in features:
        if f["images"]: flat_images.extend(f["images"])
    inputs = processor(text=batch_texts, images=flat_images if flat_images else None, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt")
    inputs["labels"] = inputs["input_ids"].clone()
    return inputs
batch = collate_fn(smoke_list[:1])  # keep everything 1: just 1 sample
print({k: v.shape if hasattr(v, 'shape') else type(v) for k,v in batch.items()})
model.train()
with torch.no_grad():
    out = model(**{k: v.to(model.device) if hasattr(v, 'to') else v for k,v in batch.items()})
print(f"Forward OK loss: {out.loss.item():.4f}")
print("If this prints, 4-bit + collator works on T4 x2")


smoke_list 2 sample label=related_papers images=0
{'input_ids': torch.Size([1, 512]), 'attention_mask': torch.Size([1, 512]), 'mm_token_type_ids': torch.Size([1, 512]), 'labels': torch.Size([1, 512])}
Forward OK loss: 4.5871
If this prints, 4-bit + collator works on T4 x2


In [8]:
# --- 6. Generate Test (10 samples, ~30s) ---
import re
label_pattern = re.compile(r"(same_figure|same_paper|related_papers|unrelated_papers)", re.IGNORECASE)
def parse_label(t):
    m=label_pattern.search(t.lower()); return m.group(1).lower() if m else "unrelated_papers"
model.eval()
preds=[]
for i in range(min(3, len(smoke_list))):
    f = smoke_list[i]
    msgs = f["messages"]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    images = f["images"] if f["images"] else None
    inputs = processor(text=[text], images=images, padding=True, return_tensors="pt")
    inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k,v in inputs.items()}
    if "pixel_values" in inputs: inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    decoded = processor.batch_decode(out[:, input_len:], skip_special_tokens=True)[0]
    pred = parse_label(decoded)
    print(f"[{i}] true={f['label']} pred={pred} raw={decoded[:80]!r}")
    preds.append(pred)
print(f"Generate OK: {preds}")
print("If this prints without OOM/AssertionError, move to Lightning A100-80GB full 8200/1800")


[0] true=related_papers pred=unrelated_papers raw='unrelated_papers'
[1] true=related_papers pred=same_paper raw='same_paper'
Generate OK: ['unrelated_papers', 'same_paper']
If this prints without OOM/AssertionError, move to Lightning A100-80GB full 8200/1800


In [9]:
# --- 7. Optional: Push smoke adapter to HF (test HF_TOKEN) ---
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo(repo_id=HF_REPO_ID, private=True, exist_ok=True)
# model.push_to_hub(HF_REPO_ID + "-smoke", private=True)
# print(f"Pushed smoke test to {HF_REPO_ID}-smoke")
print("Smoke test done — if Forward + Generate OK, your Lightning A100 run will succeed")


Smoke test done — if Forward + Generate OK, your Lightning A100 run will succeed
